# D1: sample the training data on a RunPod A100

Collects the episode files of all three splits in one run (`experiments/run.py run D1`), then checks them, writes the D0 report and uploads them to the Hugging Face Hub.

| step | cell | what it does |
|---|---|---|
| 0 | Configuration | paths, checkpoints, frame counts and shares (edit this) |
| 1 | Pod check, install | GPU, disk, `/dev/shm`; the JAX GPU environment in a venv on the volume |
| 1b | Expert throughput | optional: a short run of the expert training: seconds per update, steps per second, the cost of a greedy test, hours for 1e8 to 1e9 steps |
| 2 | Collect | the collector in the background (survives a closed browser tab); re-run the cell after a restart |
| 3 | Monitor | log tail, finished runs, frames per file |
| 4 | Checks | replay of a sample on the CPU (episodes collected on the GPU must replay there), D0 report |
| 5 | Upload | the three `.h5` files and the report to a Hugging Face dataset repo |

**Before you start.** The repository is cloned to `/workspace/AAC`, the expert's final checkpoint and its early snapshots (`..._vmap0.safetensors`, `..._vmap0_u<updates>.safetensors`, each with its `.json`/config next to it) are on the volume. The data mix, the frame counts and the snapshots are placeholders until the timing run and the D0 report (`README.md`, Data): a first run is a draft that D0 then judges. The collector appends, so more data can be added later.

**After a pod restart** run the cells from the top: finished steps are skipped (`$HJEPA_HOME/collect_done/`). A run that was killed starts again; its partial episodes stay in the file, so that share can be exceeded by up to one run's part.

In [ ]:
# --- Configuration -----------------------------------------------------------------------------------------------
import json, os, subprocess, sys, time
from pathlib import Path

VOLUME = Path("/workspace")
REPO = VOLUME / "AAC"                   # the git clone (done by hand)
VENV = VOLUME / "venvs" / "jax"         # JAX environment; on the volume, so a restarted pod reuses it
DATA_HOME = VOLUME / "data"             # HJEPA_HOME: episode files, reports, logs, checkpoints

# the expert (trained by you) and the snapshots of its training whose rollouts make the early share
EXPERT_CKPT = VOLUME / "models" / "pqn_craftax_vmap0.safetensors"
EARLY_CKPTS = [
    VOLUME / "models" / "pqn_craftax_vmap0_u0000381.safetensors",
    VOLUME / "models" / "pqn_craftax_vmap0_u0001143.safetensors",
]

# frames per split and the share of each source in them (placeholders from experiments/experiments.yaml;
# the test split holds only the final expert, its tasks come from there)
FRAMES = {"train": 2_000_000, "val": 150_000, "test": 150_000}
SHARE = {
    "train": {"expert": 0.7, "early": 0.3},
    "val": {"expert": 0.7, "early": 0.3},
    "test": {"expert": 1.0},
}

# the replay check (step 4): episodes replayed per file
REPLAY_SAMPLE = {"train": 20, "val": 10, "test": 10}

RUN_TESTS = False                       # run the repository's tests in the new environment (a few minutes)
EXPERT_OVERRIDES = []                   # KEY=value of the expert's configuration for the throughput run, e.g. ["NUM_ENVS=2048"]

In [ ]:
# --- Helpers and pod check -----------------------------------------------------------------------------------------
PYTHON = VENV / "bin" / "python"
ENV = {
    **os.environ,
    "PATH": f"{VENV / 'bin'}:{os.environ['PATH']}",   # `python` in the runner's commands is the venv's
    "HJEPA_HOME": str(DATA_HOME),
    "HDF5_USE_FILE_LOCKING": "FALSE",                 # HDF5 can fail to lock files on a network volume
    "PYTHONUNBUFFERED": "1",
}
SPEC = DATA_HOME / "spec.d1.yaml"
LOGS = DATA_HOME / "logs"


def sh(cmd, env=None, cwd=None, check=True, quiet=False):
    """Run a command and stream its output into the cell (a plain subprocess writes to the kernel's terminal).

    `quiet` keeps the output and shows only its last lines when the command fails."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(c.splitlines()[0][:100] + (" ..." if "\n" in c or len(c) > 100 else "") for c in cmd))
    try:
        proc = subprocess.Popen(cmd, env=env or ENV, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    except FileNotFoundError:
        if check:
            raise
        print(f"  ({cmd[0]} not found)")
        return 127
    lines = []
    for line in proc.stdout:
        lines.append(line)
        if not quiet:
            print(line, end="")
    code = proc.wait()
    if code and quiet:
        print("".join(lines[-30:]), end="")
    if check and code:
        raise RuntimeError(f"{cmd[0]} exited with {code}")
    return code


for command in (
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
    ["python3", "--version"],
    ["nproc"],
    ["df", "-h", str(VOLUME), "/dev/shm"],
    ["free", "-g"],
):
    sh(command, env=os.environ, check=False)

assert REPO.exists(), f"clone the repository to {REPO}"
sh(["git", "-C", REPO, "log", "-1", "--oneline"], env=os.environ)
sh(["git", "-C", REPO, "status", "--short"], env=os.environ)   # an uncommitted change is recorded in the provenance too
for path in [EXPERT_CKPT, *EARLY_CKPTS]:
    assert path.exists(), f"missing checkpoint {path}"
DATA_HOME.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(exist_ok=True)
print("\nPython 3.11 is the project's pin; /dev/shm and nproc matter for the training dataloader (num_workers 12), not here.")

In [ ]:
# --- Install the JAX GPU environment (once; skipped when the requirement files did not change) ----------------------
import hashlib

files = [REPO / "environment" / "jax-gpu-requirements.txt", REPO / "environment" / "jax-requirements.txt", REPO / "purejaxql" / "pyproject.toml"]
stamp = hashlib.sha256(b"".join(f.read_bytes() for f in files)).hexdigest()[:16]
marker = VENV / ".installed"

if not (marker.exists() and marker.read_text() == stamp):
    if not PYTHON.exists():
        sh([sys.executable, "-m", "venv", VENV], env=os.environ)
    sh([PYTHON, "-m", "pip", "install", "-q", "-U", "pip"])
    sh([PYTHON, "-m", "pip", "install", "-q", "-r", "environment/jax-gpu-requirements.txt"], cwd=REPO)
    marker.write_text(stamp)
else:
    print("environment is up to date")

sh([PYTHON, "-c", "import jax, flax, craftax; print('jax', jax.__version__, jax.devices())"])
check = subprocess.run([PYTHON, "-c", "import jax; print(jax.devices()[0].platform)"], env=ENV, capture_output=True, text=True)
assert check.stdout.strip() == "gpu", "JAX does not see the GPU: check the driver (nvidia-smi) and jax[cuda12] in the venv"

In [ ]:
# --- Optional: the repository's tests in this environment ------------------------------------------------------------
if RUN_TESTS:
    sh([PYTHON, "-m", "pytest", "-q", "purejaxql/tests", "analysis/tests", "experiments/tests"], cwd=REPO)

In [ ]:
# --- 1b. Throughput of the expert training: a short run of the real configuration ------------------------------------------
# Prints seconds per update, environment steps per second, what one greedy test costs, peak GPU memory, and the hours of runs of
# 1e8 to 1e9 steps (the run length also sets the learning-rate and epsilon schedules: decide it from this table before training).
# A few minutes, most of it compilation; snapshots and the resumable state are not included.
sh([PYTHON, REPO / "purejaxql" / "scripts" / "time_expert.py", *EXPERT_OVERRIDES], cwd=REPO / "purejaxql")

In [ ]:
# --- The plan: a copy of the experiment spec with the frame counts and shares above ---------------------------------------
import yaml   # the kernel's own python; pyyaml ships with the RunPod images (else: pip install pyyaml)

spec = yaml.safe_load((REPO / "experiments" / "experiments.yaml").read_text())
spec["collection"]["frames"] = FRAMES
spec["collection"]["share"] = SHARE
SPEC.write_text(yaml.safe_dump(spec, sort_keys=False))

RUNNER = [PYTHON, REPO / "experiments" / "run.py"]
CHECKPOINTS = ["--expert-checkpoint", EXPERT_CKPT] + [x for c in EARLY_CKPTS for x in ("--early-checkpoint", c)]
sh([*RUNNER, "plan", "D1", "--spec", SPEC, *CHECKPOINTS])

In [ ]:
# --- 2. Collect, in the background ------------------------------------------------------------------------------------------
PID = LOGS / "d1.pid"
LOG = LOGS / "d1.log"


PROC = None   # the Popen handle of this kernel's collector (a finished child stays a zombie until it is polled)


def running() -> bool:
    if PROC is not None:
        return PROC.poll() is None
    try:                       # after a kernel restart the collector, if still alive, is no longer our child
        os.kill(int(PID.read_text()), 0)
        return True
    except (FileNotFoundError, ValueError, ProcessLookupError):
        return False


if running():
    print(f"already running (pid {PID.read_text()}), follow it in the next cell")
else:
    with open(LOG, "a") as log:
        PROC = proc = subprocess.Popen([*map(str, RUNNER), "run", "D1", "--spec", str(SPEC), "--mode", "local", *map(str, CHECKPOINTS)],
                                env=ENV, cwd=REPO, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    PID.write_text(str(proc.pid))
    print(f"started pid {proc.pid}, log {LOG}")

In [ ]:
# --- 3. Monitor (run it as often as you like) -------------------------------------------------------------------------------
STATUS = """
import json, sys, h5py, numpy as np
home, frames = sys.argv[1], json.loads(sys.argv[2])
names = {"train": "craftax_expert_train", "val": "craftax_expert_val", "test": "craftax_expert_test"}
sources = {0: "expert", 1: "noised", 2: "random", 3: "early"}
for split, name in names.items():
    try:
        f = h5py.File(f"{home}/{name}.h5", "r")
    except OSError:
        print(f"{split:5s} no file yet"); continue
    with f:
        lengths, offsets = f["ep_len"][:], f["ep_offset"][:]
        source = f["source"][offsets] if len(offsets) else np.zeros(0)
        total = int(lengths.sum())
        by = ", ".join(f"{n} {int(lengths[source == c].sum()):,}" for c, n in sources.items() if (source == c).any())
        runs = json.loads(f.attrs.get("runs", "[]"))
        open_runs = sum(not r.get("finished") for r in runs)
        print(f"{split:5s} {len(lengths):6d} episodes {total:11,d} frames of {frames[split]:,} ({by})  runs {len(runs)}, unfinished {open_runs}")
"""
sh([PYTHON, "-c", STATUS, DATA_HOME, json.dumps(FRAMES)], check=False)
print("\nfinished runs:", sorted(p.name for p in (DATA_HOME / "collect_done").glob("*")) if (DATA_HOME / "collect_done").exists() else "none")
print("collector " + ("RUNNING" if running() else "NOT running"))
print("\n--- log tail ---")
sh(["tail", "-n", "12", LOG], env=os.environ, check=False)

In [ ]:
# --- Wait until it is done (optional; polls every minute and prints the last log line) --------------------------------------
while running():
    time.sleep(60)
    last = LOG.read_text().strip().splitlines()[-1:] or [""]
    print(time.strftime("%H:%M"), last[0][:160])
print("finished" if not [1 for l in LOG.read_text().splitlines() if "failed with exit code" in l] else "FAILED, see the log")

## 4. Checks

**Replay.** The planning environment runs on the CPU, the collector on the GPU, and floating point differs between them. The check replays a sample of every file on the **CPU** (`JAX_PLATFORMS=cpu`) from `seed` and the actions and compares all columns with the file. The discrete columns (full state bytes, achievements, inventory, vitals, position, ...) must be identical; if not, `NOT REPLAYABLE` and the episodes of this machine cannot be used for planning on the CPU.

**D0.** The dataset analysis (`analysis/dataset_stats.py`): the entity frequency gap, the time scale ladder, clip counts, tasks, survival. Read the WARNING lines first: they point to a wrong column or event definition.

In [ ]:
# --- Replay check on the CPU --------------------------------------------------------------------------------------------------
cpu = {**ENV, "JAX_PLATFORMS": "cpu"}
for split, name in (("train", "craftax_expert_train"), ("val", "craftax_expert_val"), ("test", "craftax_expert_test")):
    path = DATA_HOME / f"{name}.h5"
    if path.exists():
        print(f"===== {split}")
        sh([PYTHON, REPO / "purejaxql" / "scripts" / "check_replay.py", "--data", path, "--num-episodes", REPLAY_SAMPLE[split]], env=cpu, cwd=REPO / "purejaxql", check=False)

In [ ]:
# --- D0 report ---------------------------------------------------------------------------------------------------------------------
sh([*RUNNER, "run", "D0", "--spec", SPEC, "--mode", "local", "--force"], quiet=True)
report = (DATA_HOME / "reports" / "dataset_stats" / "dataset_stats.md").read_text()
print(f"\n{len(report.splitlines())} lines in {DATA_HOME / 'reports' / 'dataset_stats'}")
for line in report.splitlines():
    if line.startswith(("## ", "All 7 entities", "Without the daylight", "Check:", "Reading of the gap")) or "WARNING" in line:
        print(line[:400])

## 5. Upload to the Hugging Face Hub

A **dataset** repo with the three `.h5` files, the D0 report and the spec of this run at the root. The files stay raw HDF5 (H-JEPA's loader reads that layout), so the Hub's dataset viewer is switched off. The upload is resumable: if it is interrupted, run the cell again.

The token needs write access (huggingface.co/settings/tokens); it is read from `HF_TOKEN` or typed into the prompt, and is not stored in the notebook.

In [ ]:
# --- Upload ----------------------------------------------------------------------------------------------------------------------------
import getpass

HF_REPO_ID = "your-user/craftax-classic-d1"    # <- user/name of the dataset repo (created if it does not exist)
HF_PRIVATE = True

sh([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"], env=os.environ)
from huggingface_hub import HfApi

token = os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token (write access): ")
api = HfApi(token=token)
print("logged in as", api.whoami()["name"])
api.create_repo(HF_REPO_ID, repo_type="dataset", private=HF_PRIVATE, exist_ok=True)

# the dataset card: what is in the files, and what produced them
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
sizes = "\n".join(f"| `{p.name}` | {p.stat().st_size / 2**30:.2f} GiB |" for p in sorted(DATA_HOME.glob("*.h5")))
(DATA_HOME / "README.md").write_text(f"""---
viewer: false
---
# Craftax-Classic episodes (D1)

Episodes of the final expert and of snapshots of its training on Craftax-Classic, collected for hierarchical JEPA world models
(code and column layout: the AAC repository, commit `{commit[:10]}`). Seeds are the split: train from 0, validation from 1,000,000,
test from 2,000,000 (the test file holds only the final expert).

| file | size |
|---|---|
{sizes}

Frames per split and the share of each source (`spec.d1.yaml`): {json.dumps(FRAMES)} frames, {json.dumps(SHARE)}.
Every file records its collection runs (flags, commit, library versions, checkpoint hashes) in the HDF5 attribute `runs`.
`reports/dataset_stats/dataset_stats.md` is the analysis of these files (D0).
""")

patterns = ["*.h5", "README.md", "spec.d1.yaml", "reports/**"]
upload = getattr(api, "upload_large_folder", None) or api.upload_folder   # huggingface_hub 1.x / 2.x
upload(repo_id=HF_REPO_ID, repo_type="dataset", folder_path=str(DATA_HOME), allow_patterns=patterns)
print(f"done: https://huggingface.co/datasets/{HF_REPO_ID}")